# Parse documents with the Unstructured Transform MCP server (Agents SDK)

Real-world documents (PDFs with multi-column layouts, scanned pages, nested tables, images)
are hard to feed to a language model directly. [Unstructured](https://unstructured.io) solves
this with **Transform**, a hosted service that turns 60+ file types into clean, structured
content (markdown, Element JSON, HTML, or text).

Unstructured Transform is available as a remote [Model Context Protocol (MCP)](https://modelcontextprotocol.io)
server. This recipe connects that server to the [**OpenAI Agents SDK**](https://openai.github.io/openai-agents-python/)
as a client-managed MCP server: the SDK opens the connection, discovers the Transform tools,
and hands them to an `Agent` that decides when to call them. The agent parses a complex,
table-heavy PDF ("Attention Is All You Need") into clean markdown, and then a second agent
answers questions grounded in the parsed content.

> This is the Agents SDK counterpart to the
> [Responses API recipe](https://cookbook.openai.com/examples/mcp/parse_documents_with_unstructured_transform_mcp).
> Same document, same questions. The difference is that here the SDK manages the MCP connection
> from your process (`mcp_servers=[...]`) rather than the call running hosted server-side.

## 1. Prerequisites

The OpenAI Agents SDK requires **Python 3.10 or newer**. On an older interpreter you will see
`TypeError: Unable to evaluate type annotation 'float | None'` when importing `agents`. If you
run this in Jupyter, make sure the selected kernel points at a Python 3.10+ environment.

You need two API keys, both read from environment variables (never hard-code secrets in a notebook):

- **`OPENAI_API_KEY`** from <https://platform.openai.com/api-keys>.
- **`UNSTRUCTURED_API_KEY`** sign up at [Unstructured](https://unstructured.io/?modal=try-for-free), then log in to get your key.

```bash
export OPENAI_API_KEY="sk-..."
export UNSTRUCTURED_API_KEY="..."
```

In [ ]:
%pip install --upgrade openai-agents requests

In [ ]:
import asyncio
import json
import os
import re
from getpass import getpass

import requests
from agents import Agent, Runner, function_tool
from agents.mcp import MCPServerStreamableHttp

if "UNSTRUCTURED_API_KEY" not in os.environ:
    os.environ["UNSTRUCTURED_API_KEY"] = getpass("Enter your Unstructured API key: ")
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

# The Agents SDK reads OPENAI_API_KEY from the environment automatically.
# The hosted Unstructured Transform MCP server speaks streamable HTTP.
TRANSFORM_MCP_URL = "https://mcp.transform.unstructured.io/"
MODEL = "gpt-5"

## 2. Connect the Transform MCP server

With the Agents SDK you connect to a remote MCP server through `MCPServerStreamableHttp`.
The Transform server authenticates with your Unstructured API key passed as a **bearer token**
in the `Authorization` header, which you provide inside `params`.

In a script you would normally use `async with MCPServerStreamableHttp(...) as server:` so the
connection is opened and closed for you. In a notebook it is more convenient to call
`await server.connect()` once and keep the server available across cells, then call
`await server.cleanup()` at the end. `cache_tools_list=True` caches the `tools/list` result so
the SDK does not re-fetch it on every run.

In [ ]:
transform_server = MCPServerStreamableHttp(
    name="unstructured_transform",
    params={
        "url": TRANSFORM_MCP_URL,
        "headers": {"Authorization": f"Bearer {os.environ['UNSTRUCTURED_API_KEY']}"},
        "timeout": 30,
    },
    cache_tools_list=True,
    max_retry_attempts=3,
)

await transform_server.connect()

# Confirm connectivity by listing the tools Transform exposes.
tools = await transform_server.list_tools()
print("Tools exposed by the Unstructured Transform MCP server:")
for tool in tools:
    print(f"  - {tool.name}")

You should see the four Transform tools:

- `request_file_upload_url` mints a signed URL to upload a local file.
- `transform_files` submits 1-10 inputs (public URLs or uploads) as one async job.
- `check_transform_status` polls a job until it completes.
- `get_transform_results` retrieves the rendered output (markdown / JSON / HTML / text).

Because our document is already reachable at a public `https://` URL, we can skip the upload
step and hand the URL straight to `transform_files`.

## 3. Build the agent

Transform runs the parse as an **asynchronous job**, so the agent has to poll
`check_transform_status` until it is done. To keep it from hammering the server, we give it a
local `wait_seconds` function tool it can call between checks. Unlike the MCP tools (which the
SDK routes to the Transform server), this tool runs in your own process. The Agents SDK
executes function tools automatically, so we just decorate a coroutine with `@function_tool`
and add it to the agent's `tools` list alongside the `mcp_servers`.

In [ ]:
@function_tool
async def wait_seconds(seconds: int) -> str:
    """Wait the given number of seconds before checking the job status again."""
    await asyncio.sleep(seconds)
    return f"Waited {seconds} seconds."

Now attach the connected server and the wait tool to an `Agent`. The agent holds the Transform
tools plus `wait_seconds` and decides when to call them, driving the full async pipeline on its
own: submit the job, wait and poll until it completes, then fetch the results.

In [ ]:
doc_agent = Agent(
    name="Document parser",
    model=MODEL,
    instructions=(
        "You parse documents using the Unstructured Transform MCP tools. "
        "When asked to parse a file, submit the job with transform_files, then poll "
        "check_transform_status, calling wait_seconds(10) between each check, until the "
        "status is COMPLETED. Then retrieve the results with get_transform_results. "
        "Be concise in your final report."
    ),
    tools=[wait_seconds],
    mcp_servers=[transform_server],
)

## 4. Parse a complex PDF

Hand the agent a public PDF URL and let it run the whole Transform loop within a single
`Runner.run` call. The agent will call `transform_files`, poll with `check_transform_status`,
and then call `get_transform_results`.

In [ ]:
PDF_URL = "https://arxiv.org/pdf/1706.03762"  # "Attention Is All You Need"

parse_result = await Runner.run(
    doc_agent,
    (
        "Use the Unstructured Transform tools to parse the PDF at "
        f"{PDF_URL} into markdown. Submit the job, wait for it to complete, then "
        "retrieve the markdown results. Report how many elements were extracted "
        "and the download URL for the parsed markdown."
    ),
    # Transform runs asynchronously, so the agent polls check_transform_status
    # in a loop. Each poll is one turn, so raise the default max_turns (10).
    max_turns=40,
)

print(parse_result.final_output)

### Fetch the rendered output

`get_transform_results` returns each file **out of band**: instead of dumping a large document
into the model's context, it returns a short-lived, signed `download_url`. This keeps the
agent's context lean. To reason over the full content ourselves, we recover that URL from the
run's tool-call outputs and download it directly.

`Runner.run` returns a `RunResult` whose `new_items` list captures every tool call and tool
output. We scan the serialized tool outputs for the signed markdown URL.

In [ ]:
def extract_markdown_url(result):
    """Recover the Transform markdown download URL from the run's tool-call outputs.

    ``get_transform_results`` embeds a signed ``download_url`` in its tool output. We scan the
    serialized ``new_items`` for it rather than depending on a specific field path.
    """
    raw_items = [
        getattr(item, "raw_item", None)
        for item in result.new_items
        if getattr(item, "raw_item", None) is not None
    ]
    blob = json.dumps(
        raw_items,
        default=lambda o: o.model_dump() if hasattr(o, "model_dump") else str(o),
    )
    matches = re.findall(
        r"https://mcp\.transform\.unstructured\.io/output/\S+?\.md[^\"\\ ]*",
        blob,
    )
    if not matches:
        raise RuntimeError("No Transform markdown download URL found in the run items.")
    return matches[0]


md_url = extract_markdown_url(parse_result)
parsed_markdown = requests.get(md_url, timeout=60).text
print(parsed_markdown[:1000])

Notice that tables come back as clean HTML `<table>` blocks embedded in the markdown, so
structure like the BLEU-score comparison table survives, exactly the kind of content that
naive PDF text extraction mangles.

## 5. Ask questions over the parsed content

With clean markdown in hand, a second agent answers questions grounded in the document. This
agent has no tools: it just reasons over the text we pass it. We keep it honest by instructing
it to answer only from the provided content.

In [ ]:
qa_agent = Agent(
    name="Grounded QA",
    model=MODEL,
    instructions=(
        "Answer only from the provided document. "
        "Cite the table or section you used."
    ),
)

qa_result = await Runner.run(
    qa_agent,
    (
        "Here is a research paper parsed to markdown:\n\n"
        f"{parsed_markdown}\n\n"
        "1) What BLEU scores did the Transformer (big) model achieve on the "
        "EN-DE and EN-FR newstest2014 tasks?\n"
        "2) According to the model-architecture table, what is the maximum path "
        "length for a self-attention layer?"
    ),
)

print(qa_result.final_output)

The agent reads the answers straight out of the parsed tables. For example, Transformer (big)
scores **28.4 BLEU (EN-DE)** and **41.8 BLEU (EN-FR)** from Table 2, and a self-attention layer
has a maximum path length of **O(1)** per Table 1.

## 6. Clean up the connection

Because we opened the server with `await server.connect()`, we close it explicitly when done.
(The `async with` form does this for you automatically.)

In [ ]:
await transform_server.cleanup()

## 7. Best practices

- **Prefer `async with` in scripts.** `async with MCPServerStreamableHttp(...) as server:`
  opens and closes the connection for you. The explicit `connect()` / `cleanup()` used here is
  a notebook convenience so the server persists across cells.
- **Cache the tool list.** `cache_tools_list=True` avoids re-fetching `tools/list` on every run.
  Call `server.invalidate_tools_cache()` if the server's tools change.
- **Restrict the tool surface.** Pass a `tool_filter` to the server (for example
  `create_static_tool_filter(allowed_tool_names=[...])`) to expose only the tools a task needs.
- **Mind the limits.** Transform accepts up to 50 MB per file, 10 files per request, and 5
  concurrent jobs. For local files, have the agent mint an upload URL with
  `request_file_upload_url` first.
- **Pick your output format.** `get_transform_results` can render the same job as markdown,
  Element JSON, HTML, or plain text. Ask for Element JSON when you want to build a retrieval
  index over typed elements.

## Conclusion

By connecting Unstructured Transform as a client-managed MCP server, an OpenAI Agents SDK agent
can parse messy, real-world documents on demand and reason over the clean result, with no
custom extraction pipeline. From here you could:

- Swap in your own PDFs, DOCX, PPTX, XLSX, emails, or images.
- Request **Element JSON** and build a retrieval index over the typed elements.
- Combine the Transform server with other tools (web search, file search) or hand off between
  specialized agents in a larger workflow.

### Resources

- [OpenAI Agents SDK: MCP](https://openai.github.io/openai-agents-python/mcp/)
- [Unstructured Transform overview](https://docs.unstructured.io/transform/overview)
- [Model Context Protocol](https://modelcontextprotocol.io)